In [3]:
pip install pytrends

In [10]:
import pandas as pd
from pytrends.request import TrendReq
import time

# -----------------------------
# 설정
# -----------------------------

INPUT_FILE = "skin_surgery_40.xlsx"
OUTPUT_FILE = "google_trends_skin_surgery_lang_by_country.xlsx"

# 각 나라가 어느 언어 컬럼을 사용할지 매핑
COUNTRY_CONFIG = {
    "US": {"geo": "US", "col": "영어"},
    "JP": {"geo": "JP", "col": "일본어"},
    "KR": {"geo": "KR", "col": "한국어"},
    # 중국(Baidu)는 나중에 따로 처리
}

MONTHS = 24      # 최근 24개월
CHUNK = 5        # Google Trends 한 번에 최대 키워드 수(5개)

# -----------------------------
# 1) 엑셀에서 언어별 키워드 불러오기
# -----------------------------
df = pd.read_excel(INPUT_FILE)

def get_keywords_by_lang(col_name: str):
    """언어별 컬럼에서 키워드 리스트 추출 (공백/NaN 제거 + 순서 유지)"""
    vals = df[col_name].dropna().astype(str).str.strip().tolist()
    # 완전 동일한 문자열은 한 번만 남기기 (안전용)
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return out

# 언어별 키워드 미리 확인
eng_keywords = get_keywords_by_lang("영어")
jp_keywords  = get_keywords_by_lang("일본어")
kr_keywords  = get_keywords_by_lang("한국어")

print("✅ 영어 키워드 수:", len(eng_keywords), "예시:", eng_keywords[:10])
print("✅ 일본어 키워드 수:", len(jp_keywords), "예시:", jp_keywords[:10])
print("✅ 한국어 키워드 수:", len(kr_keywords), "예시:", kr_keywords[:10])

# -----------------------------
# 2) Pytrends 객체
# -----------------------------
pytrends = TrendReq(hl='en-US', tz=540)

# -----------------------------
# 3) 리스트를 n개씩 나누기
# -----------------------------
def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# -----------------------------
# 4) 국가별 + 언어별로 트렌드 수집
# -----------------------------
all_country_frames = []

for country_code, cfg in COUNTRY_CONFIG.items():
    geo = cfg["geo"]
    col = cfg["col"]

    if col == "영어":
        kw_list = eng_keywords
    elif col == "일본어":
        kw_list = jp_keywords
    elif col == "한국어":
        kw_list = kr_keywords
    else:
        continue

    print("\n==============================")
    print(f"   🌍 국가: {country_code} / 언어 컬럼: {col}")
    print("==============================")
    print(f"   총 키워드 수: {len(kw_list)}개")

    country_df_all = pd.DataFrame()

    for idx, group in enumerate(chunks(kw_list, CHUNK)):
        print(f"\n[{country_code}] 키워드 그룹 {idx+1} : {group}")

        if not group:
            continue

        # 1) Pytrends 요청
        try:
            pytrends.build_payload(group, timeframe="today 5-y", geo=geo)
            df_trend = pytrends.interest_over_time()
        except Exception as e:
            print(f"   ❌ pytrends 호출 오류 (group={group}): {e}")
            continue

        if df_trend.empty:
            print("   ⚠ 결과가 비어 있음 → 스킵")
            continue

        # isPartial 제거
        if "isPartial" in df_trend.columns:
            df_trend = df_trend.drop(columns=["isPartial"])

        # 2) 월별 리샘플링 (M=월말 기준)
        df_monthly = df_trend.resample("M").mean()

        # 3) 최근 24개월만 사용
        df_24m = df_monthly.tail(MONTHS)

        # 4) 컬럼명에 국가 코드 suffix 붙이기
        #    예: "acne_US", "ニキビ_JP", "여드름_KR"
        df_24m = df_24m.add_suffix(f"_{country_code}")

        # 5) 국가별 DataFrame에 병합
        if country_df_all.empty:
            country_df_all = df_24m
        else:
            country_df_all = country_df_all.join(df_24m, how="outer")

        # 구글에 과도한 연속 요청 방지
        time.sleep(1.0)

    all_country_frames.append(country_df_all)

# -----------------------------
# 5) 모든 국가 데이터 합치기
# -----------------------------
final_df = all_country_frames[0]
for df_c in all_country_frames[1:]:
    final_df = final_df.join(df_c, how="outer")

# -----------------------------
# 6) 엑셀 저장
# -----------------------------
final_df.to_excel(OUTPUT_FILE)
print(f"\n🎉 저장 완료 → {OUTPUT_FILE}")

✅ 영어 키워드 수: 40 예시: ['acne', 'acne scars', 'pigmentation', 'melasma', 'large pores', 'skin sagging', 'wrinkles', 'fine lines', 'nasolabial fold', 'dark circles']
✅ 일본어 키워드 수: 39 예시: ['ニキビ', 'ニキビ跡', '色素沈着', '肝斑', '毛穴', '顔のたるみ', 'しわ', '小じわ', 'ほうれい線', 'クマ']
✅ 한국어 키워드 수: 40 예시: ['여드름', '여드름 흉터', '색소침착', '기미 / 멜라스마', '모공', '피부 처짐', '주름', '잔주름 / 미세주름', '팔자주름', '다크서클']

   🌍 국가: US / 언어 컬럼: 영어
   총 키워드 수: 40개

[US] 키워드 그룹 1 : ['acne', 'acne scars', 'pigmentation', 'melasma', 'large pores']


C:\Users\user\anaconda3\Lib\site-packages\pytrends\request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_8528\849493695.py:109: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df_trend.resample("M").mean()



[US] 키워드 그룹 2 : ['skin sagging', 'wrinkles', 'fine lines', 'nasolabial fold', 'dark circles']


C:\Users\user\anaconda3\Lib\site-packages\pytrends\request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_8528\849493695.py:109: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df_trend.resample("M").mean()



[US] 키워드 그룹 3 : ['dull skin', 'uneven skin tone', 'redness', 'double chin', 'jawline sagging']


...


[KR] 키워드 그룹 7 : ['사각턱 / 각진 턱', '무턱 / 턱 짧음', '턱 뒤로 들어감', '긴 얼굴형', '둥근 얼굴형']


C:\Users\user\anaconda3\Lib\site-packages\pytrends\request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_8528\849493695.py:109: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df_trend.resample("M").mean()



[KR] 키워드 그룹 8 : ['광대 돌출', '광대 넓음', '얇은 입술', '잇몸 웃음(거미스마일)', '입술 비대칭']


C:\Users\user\anaconda3\Lib\site-packages\pytrends\request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
C:\Users\user\AppData\Local\Temp\ipykernel_8528\849493695.py:109: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_monthly = df_trend.resample("M").mean()



🎉 저장 완료 → google_trends_skin_surgery_lang_by_country.xlsx


In [9]:
import pandas as pd

FILE = "skin_surgery_40.xlsx"

df = pd.read_excel(FILE)

print("📂 전체 컬럼:")
print(df.columns)
print()

print("📏 전체 행 수:", len(df))
print(df.head())

print("\n🇺🇸 영어 키워드 개수:", df["영어"].notna().sum())
print("예시:", df["영어"].head().tolist())

print("\n🇯🇵 일본어 키워드 개수:", df["일본어"].notna().sum())
print("예시:", df["일본어"].head().tolist())

print("\n🇰🇷 한국어 키워드 개수:", df["한국어"].notna().sum())
print("예시:", df["한국어"].head().tolist())

📂 전체 컬럼:
Index(['분류', '영어', '일본어', '중국어', '한국어'], dtype='object')

📏 전체 행 수: 40
     분류            영어   일본어   중국어        한국어
0  skin          acne   ニキビ    痤疮        여드름
1  skin    acne scars  ニキビ跡    痘疤     여드름 흉터
2  skin  pigmentation  色素沈着  色素沉着       색소침착
3  skin       melasma    肝斑   黄褐斑  기미 / 멜라스마
4  skin   large pores    毛穴  毛孔粗大         모공

🇺🇸 영어 키워드 개수: 40
예시: ['acne', 'acne scars', 'pigmentation', 'melasma', 'large pores']

🇯🇵 일본어 키워드 개수: 40
예시: ['ニキビ', 'ニキビ跡', '色素沈着', '肝斑', '毛穴']

🇰🇷 한국어 키워드 개수: 40
예시: ['여드름', '여드름 흉터', '색소침착', '기미 / 멜라스마', '모공']
